# 02 — Text Cleaning & Normalization

**Learning objective.** Clean noisy text without destroying task-relevant information, using explicit and testable normalization rules.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**raw text → selected normalization → canonical text → cleaner feature space**

The key question is not “which API do I call?” but **which representation changes next when I change a control?**

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Lowercase text | case variants collapse | vocabulary shrinks; entity/case signal may disappear |
| Replace emails/URLs with placeholders | literal identities collapse into a type | memorization decreases while presence signal remains |
| Remove punctuation aggressively | feature space shrinks | sentiment, IDs and negation cues can be damaged |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. Will lowercasing help `Apple` vs `apple` entity recognition?
2. What happens if every order ID is kept as a unique token?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Use normalization to remove nuisance variation that is not part of the target.

### When not to use / caution
Do not clean because text 'looks messy'; preserve any symbol whose meaning may matter.

### Debugging lens
Compare raw and normalized examples side by side and ask: which evidence was intentionally removed, merged or preserved?

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


## Principle: normalization is a task decision
Lower-casing may help a topic classifier but destroy signals for an entity recognizer. Removing punctuation can simplify bag-of-words while harming sentiment (`!`) or product identifiers (`RTX-5090`). Every transformation must have a reason.

In [2]:
import unicodedata, html
samples = pd.Series([
 '  GREAT phone!!!  ',
 '<b>Refund</b> delayed — email me at user@example.com',
 'Visit https://example.com now…',
 'Price: ₹10,000\nAvailable TODAY.'
])

def normalize(text):
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+', '<URL>', text)
    text = re.sub(r'[\w.+-]+@[\w.-]+\.\w+', '<EMAIL>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
out = pd.DataFrame({'raw':samples,'normalized':samples.map(normalize)})
out

                                                    raw                            normalized
0                                      GREAT phone!!!                          GREAT phone!!!
1  <b>Refund</b> delayed — email me at user@example.com  Refund delayed — email me at <EMAIL>
2                        Visit https://example.com now…                    Visit <URL> now...
3                      Price: ₹10,000\nAvailable TODAY.       Price: ₹10,000 Available TODAY.

In [3]:
checks = {
 'URL preserved as semantic placeholder': '<URL>' in normalize(samples.iloc[2]),
 'email protected as placeholder': '<EMAIL>' in normalize(samples.iloc[1]),
 'whitespace collapsed': '  ' not in normalize(samples.iloc[0]),
}
checks

{'URL preserved as semantic placeholder': True,
 'email protected as placeholder': True,
 'whitespace collapsed': True}

---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Treat normalization as a model/data contract
- Preserve semantic placeholders rather than blindly deleting evidence